# 🇹🇳 Tunisian Arabizi Assistant — clean retrain (anti-overfit + real-data anchor)

Standard HuggingFace **QLoRA** (transformers + peft + bitsandbytes). This version fixes the
last run, which **overfit** (loss started at 0.13) and produced fluent-looking **gibberish**.

**What changed**
- **Real-data anchor:** mixes `real_pairs.jsonl` (REAL human Arabizi from the lexicon — the model
  can't invent words) with your conversational pairs, instead of synthetic-only.
- **Anti-overfit:** 1 epoch, LR 1e-4, LoRA r=8, **no** conversation upsampling.
- **Honest metric:** headline is **real-word rate** (vs the 17k lexicon), not the surface
  'dialect rate' that read 96% on gibberish.

## Setup (one-time)
1. **+ Add Data → New Dataset** → upload `cs_pairs.jsonl`, `real_pairs.jsonl`, `eval_set.jsonl`,
   **and `lexicon.jsonl`** (from `rag/`, needed for the real-word metric).
2. Right panel: **Accelerator = GPU T4 x2**, **Internet = ON**.
3. **Run All**. Qwen2.5-7B on a T4 ≈ 1.5–2.5 h.


## 1. Install (pinned, no Unsloth)

In [ ]:
%%capture
!pip install -q -U "transformers<5" "peft>=0.12" "bitsandbytes>=0.43" accelerate datasets


## 2. Config

In [ ]:
MODEL_NAME      = 'Qwen/Qwen2.5-7B-Instruct'   # bigger brain -> more coherent
#  faster/smaller test: 'Qwen/Qwen2.5-3B-Instruct' (then BATCH=2, GRAD_ACCUM=4)
MAX_SEQ_LEN     = 1024

# --- data mix ---------------------------------------------------------------
CONV_UPSAMPLE   = 1       # NO upsampling (3x is what overfit the synthetic pairs last run)
REAL_SAMPLE     = 4000    # real human Arabizi pairs -> anchor correct words/spelling

# --- anti-overfit -----------------------------------------------------------
EPOCHS          = 1       # 1 epoch (loss started at 0.13 last run = already memorized)
LR              = 1e-4    # gentler -> learn words, don't memorize phrasings
LORA_R          = 8       # smaller adapter -> less capacity to overfit
BATCH           = 1       # 7B on a single T4 needs batch 1
GRAD_ACCUM      = 16      # effective batch = 16
EVAL_MAX_NEW    = 200

SYSTEM = ('Enti assistant tunsi (service client w 7adith 3am). Jaweb DIMA bel derja tounsiya '
          'bel arabizi (7ourouf latiniya w arqam), b tari9a tabi3iya w 9sira. Ken el user yekteb '
          'bel 3arbi wala faransi wala anglais, efhem w jaweb bel arabizi. Ken talbou translation, '
          'a3mel el li talbou.')
print('config ready ->', MODEL_NAME)


## 3. Load + mix the data

In [ ]:
import json, glob, random, re
random.seed(42)
def find(name):
    h = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not h: raise FileNotFoundError(f'{name} not found under /kaggle/input — upload your dataset')
    return h[0]
def load_jsonl(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

_AR = re.compile(r'[\u0600-\u06ff]')
def _arabizi_out(t):
    t = t or ''
    if _AR.search(t): return False                          # arabic script -> skip
    return bool(re.search(r"[a-z][3579]|[3579][a-z]|\b[3579]\b", t.lower()))  # has arabizi number-letters

conv = load_jsonl(find('cs_pairs.jsonl'))
# OPTIONAL real conversational gold (dataset/tools/build_real_conv.py). Best coherence data
# if you have it — folded into the conversational pool. Skipped silently if not uploaded.
try:
    rc = load_jsonl(find('real_conv_pairs.jsonl'))
    conv = conv + [{'instruction': r['instruction'], 'output': r['output']} for r in rc]
    print(f'+ real_conv {len(rc)} real comment->reply pairs folded into conversational pool')
except FileNotFoundError:
    pass
# real human Arabizi anchor: keep ONLY pairs whose OUTPUT is Arabizi (English->Derja
# generation). The comprehend/vocab pairs output English -> they'd teach the model to
# answer in English, which is the opposite of what we want.
real_all = load_jsonl(find('real_pairs.jsonl'))
real = [{'instruction': r['instruction'], 'output': r['output']} for r in real_all if _arabizi_out(r['output'])]
random.shuffle(real); real = real[:REAL_SAMPLE]
rows = conv*CONV_UPSAMPLE + real
train_rows = [{'instruction': r['instruction'], 'output': r['output']} for r in rows]
random.shuffle(train_rows)
print(f'conversational {len(conv)} x{CONV_UPSAMPLE} | real-anchor {len(real)} | TOTAL train {len(train_rows)}')


## 4. Load model in 4-bit + attach LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                             device_map='auto', torch_dtype=torch.float16)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_R*2, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 5. Chat helper (used for baseline + final test)

In [ ]:
def generate(user_msg, system=SYSTEM, max_new_tokens=EVAL_MAX_NEW, temperature=0.7):
    model.eval()
    msgs = [{'role':'system','content':system},{'role':'user','content':user_msg}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids),
            max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=0.9,
            repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


## 6. Dialect-rate metric (MSA-leakage detector)

In [ ]:
import re
TUN = set('barcha famma chnowa chnoua 9addech 9adech kifech 3lech win wa9tech ya5i 5ouya mte3 mta3 '
  'bch taw tawa ken kima brabi 3aslama 3aslema marhba chwaya barka fissa3 zin behi bahi mli7 3andi '
  '3andou 3andek hethi hetha hakka haka sa7a 3aychek yezzi 9a3ed mch mouch ma3andich n7eb t7eb 9olli '
  'ya3ni fel lel el enti ena houwa hia 9ahwa ghodwa lyoum wala 5dma 5edma dar bnin tounsi'.split())
FR = set('livraison prix commande merci bonjour stock weekend promo garantie couleur taille'.split())
MSA = set('hadha hadhihi alladhi allati sawfa laysa kayfa limadha 3indama ladhalika lakinna jiddan '
  'kathiran yumkinu yajibu na7nu inna sayakun dhalika tilka hunaka faqat aydan ladayna lan lam qad'.split())
MSA_AR = ['الذي','التي','سوف','ليس','كيف','عندما','لذلك','يمكن','يجب','نحن','جدا','هذا','هذه','ذلك']
_AR=re.compile(r'[\u0600-\u06ff]'); _NUM=re.compile(r"[a-z][3-9'][a-z]",re.I); _W=re.compile(r"[a-z0-9'7359]+",re.I)
def score_text(t):
    t=(t or '').strip(); toks=set(w.lower() for w in _W.findall(t))
    tun=len(toks&TUN)+len(toks&FR)+len(_NUM.findall(t.lower())); msa=len(toks&MSA)+sum(t.count(m) for m in MSA_AR)
    if _AR.search(t) and not (toks&TUN): return 'arabic_script'
    if tun==0 and msa==0: return 'unknown'
    if msa>0 and tun==0: return 'msa_leak'
    if tun>0 and msa==0: return 'tunisian'
    return 'tunisian' if tun>=2*msa else 'mixed'
def dialect_report(preds, tag=''):
    from collections import Counter
    c=Counter(score_text(p) for p in preds); n=len(preds) or 1
    print(f'--- {tag} (n={len(preds)}) ---')
    for k in ['tunisian','mixed','msa_leak','arabic_script','unknown']: print(f'  {k:14}: {c.get(k,0):4} ({c.get(k,0)/n:.0%})')
    r=c.get('tunisian',0)/n; print(f'  >> DIALECT RATE: {r:.0%}'); return r


## 6b. Real-word rate (coherence) — the metric that actually catches gibberish
Scores each output against the **17k lexicon** (real Arabizi words + their example sentences).
Skeleton matching tolerates spelling variants; invented words (`najje9`, `njaafolek`) stay OOV.

In [ ]:
import json, re
def _norm(t):
    t=t.lower(); t=re.sub(r"[^\w'89]",'',t); t=re.sub(r'(.)\1{2,}',r'\1',t)
    return t.replace('sh','ch').replace('kh','5')
def _skel(k): return re.sub(r"[aeiouy']",'',k)
_FUNC=set(('el la w f fi 3la 3al men mel l b bel ma mch mouch ena enti enta houwa hia a7na entouma houma '
  'hethi hetha haka hakka ken kima taw tawa bch barcha barka chwaya yezzi famma 9addech kifech chnowa win '
  '3lech ya5i brabi 3aslema marhba sa7a 3aychek 9a3ed n7eb t7eb 9olli ya3ni wala ama 3andi 3andou 3andek mte3 '
  'mta3 ki bla zeyed akther a9all a7sen behi mli7 zin 5ater tounes tounsi lyoum ghodwa 8odwa sba7 msa nhar ey '
  'la2 7atta inchallah rabbi 7amdoulah dt tnd livraison prix commande merci stock promo').split())
def build_vocab():
    try: lex=find('lexicon.jsonl')
    except Exception: print('  (lexicon.jsonl not uploaded -> real-word metric disabled)'); return None,None
    vocab=set(_FUNC)
    for l in open(lex,encoding='utf-8'):
        try: e=json.loads(l)
        except: continue
        for v in (e.get('arabizi_variants') or []): vocab.add(_norm(v))
        for wd in re.split(r'\s+', e.get('example_arabizi') or ''):
            k=_norm(wd);
            if len(k)>=2: vocab.add(k)
    vocab.discard(''); skels={_skel(v) for v in vocab if len(_skel(v))>=2}
    return vocab, skels
_VOCAB,_SKELS=build_vocab()
_WORD=re.compile(r"[A-Za-z0-9'][A-Za-z0-9']*")
def realword_text(t):
    if _VOCAB is None: return None
    toks=[w for w in _WORD.findall(t or '') if not w.isdigit()]; 
    if not toks: return 1.0
    ok=0
    for w in toks:
        k=_norm(w);
        if len(k)<2: ok+=1; continue
        sk=_skel(k)
        if k in _VOCAB or (len(sk)>=2 and sk in _SKELS): ok+=1
    return ok/len(toks)
def realword_report(preds, tag=''):
    if _VOCAB is None: return None
    rs=[realword_text(p) for p in preds]; m=sum(rs)/len(rs)
    print(f'  >> REAL-WORD RATE [{tag}]: {m:.0%}'); return m


## 7. BASELINE — *before* training (real-word rate + dialect rate)

In [ ]:
eval_rows = load_jsonl(find('eval_set.jsonl'))
print('eval items:', len(eval_rows))
base_preds = [generate(r['instruction']) for r in eval_rows]
base_rate = dialect_report(base_preds, 'BASELINE (before fine-tuning)')
base_rw   = realword_report(base_preds, 'BASELINE')


## 8. Train (QLoRA, standard HF Trainer)

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

def fmt(r):
    text = tokenizer.apply_chat_template(
        [{'role':'system','content':SYSTEM},{'role':'user','content':r['instruction']},
         {'role':'assistant','content':r['output']}], tokenize=False)
    enc = tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN)
    enc['labels'] = enc['input_ids'].copy()
    return enc

base_ds = Dataset.from_list(train_rows)
train_ds = base_ds.map(fmt, remove_columns=base_ds.column_names)
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

args = TrainingArguments(
    output_dir='outputs', per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS, learning_rate=LR, warmup_steps=10, logging_steps=20,
    fp16=True, optim='paged_adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine',
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    save_strategy='no', report_to='none', seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
model.config.use_cache = False
trainer.train()


## 9. Save the LoRA adapter (download from the Output tab)

In [ ]:
model.save_pretrained('/kaggle/working/tunisian_lora')
tokenizer.save_pretrained('/kaggle/working/tunisian_lora')
print('saved -> /kaggle/working/tunisian_lora')


## 10. AFTER — dialect rate *after* training (the dataset-efficiency result)

In [ ]:
after_preds = [generate(r['instruction']) for r in eval_rows]
after_rate = dialect_report(after_preds, 'AFTER fine-tuning')
after_rw   = realword_report(after_preds, 'AFTER')
print(f'\n=== RESULT ===')
if base_rw is not None:
    print(f'REAL-WORD RATE  BEFORE: {base_rw:.0%}   AFTER: {after_rw:.0%}   ({(after_rw-base_rw)*100:+.0f} pts)  <- the one that matters')
print(f'dialect rate    BEFORE: {base_rate:.0%}   AFTER: {after_rate:.0%}   (surface marker, saturates easily)')
print('\n--- 8 sample before/after (NEW should be coherent, not just Arabizi-looking) ---')
for r,b,a in list(zip(eval_rows, base_preds, after_preds))[:8]:
    print('Q  :', r['instruction'][:70]); print('OLD:', b[:90]); print('NEW:', a[:90]); print()


## 11. Talk to your model 🇹🇳 (understands Arabizi, Arabic letters, French, English)

In [ ]:
for msg in ['3aslema chna7welek?','قداش تمن التوصيل لصفاقس؟','give me a healthy breakfast idea',
            '9olli nokta tdha7ek','a7kili 7keya 9sira 3la el sabr']:
    print('🧑', msg); print('🤖', generate(msg), '\n')


In [ ]:
# >>> your turn: change this and re-run <<<
print(generate('3andi mochkla fel commande mte3i, chnowa na3mel?'))


## 12. Read the result
- **REAL-WORD RATE** is the number that matters. If AFTER ≥ BEFORE *and* the sample answers read
  coherent, the data worked. If AFTER **drops** below BEFORE, the fine-tune is hurting →
  **ship the base model + serving grounding instead** (that was the whole finding).
- Dialect rate is kept only as a surface check — it saturates near 96% on gibberish, so don't trust it alone.
- **Download** `/kaggle/working/tunisian_lora` (Output tab) and point `serving/` at it.
- Next levers if coherence is still weak: more REAL conversational data (PII-stripped scrapes →
  transliterate), and keep `candidates`/best-of-N grounding ON at serving time.
